In [1]:
from pyconll.conllu import conllu
import pandas as pd
import os
from dotenv import load_dotenv

One issue is that the key is not consistent between sentences to be able to join on the data in the meta.tsv files. They rely on the first sentence in the speech being labelled with 'newdoc id' and there are different IDs for individual utterances, but in order to classify at the sentence level we must associate each utterance with the speaker/party/orientation and therefore the class label. We need to create a consistent join key for each sentence.

In [2]:
def create_join_keys(file):
    current_id = ''
    for sentence in file:
        if sentence.meta.get('newdoc id', None) is not None:
            current_id = sentence.meta['newdoc id']
            sentence.meta.update({'join key': current_id})
        else:
            sentence.meta.update({'join key': current_id})
    return file

In [3]:
def file_to_token_array_dataframe(file):
    sentence_array = []
    for sentence in file:
        token_array = []
        lemma_array = []
        for token in sentence.tokens:
            token_array.append(token.form.lower())
            lemma_array.append(token.lemma.lower())
        sentence_array.append({'tokens': token_array, 'lemmas': lemma_array, 'join_key': sentence.meta['join key'], 'length': len(token_array)})
    return pd.DataFrame(sentence_array)

In [4]:
def process_metadata(file_path):
    return pd.read_csv(file_path, sep='\t')


In [5]:
def get_metadata_dataframe(file_path):
    df = pd.read_csv(file_path, sep='\t')
    df = df.rename({'ID': 'join_key'}, axis='columns')
    return df[['join_key', 'Date', 'Speaker_role', 'Speaker_MP', 'Speaker_minister', 'Speaker_party', 'Party_orientation', 'Speaker_ID', 'Speaker_name']]

In [6]:
def create_joint_dataframe(text_dataframe, metadata_dataframe):
    return text_dataframe.merge(metadata_dataframe, on='join_key', how='outer')

In [7]:
def get_dataset_paths(year = None):
    load_dotenv()
    path = os.environ['DATASET_PATH']
    dataset_paths = []
    walk = os.walk(path)
    for folder in walk:
        if year is not None and year not in folder[0]:
            continue
        if folder[0] != path:
            files = os.listdir(folder[0])
            for file in files:
                if '.conllu' in file:
                    filename_root = file[0:file.find('.conllu')]
                    meta = filename_root + '-meta-en.tsv'
                    dataset_paths.append({'conllu': os.path.join(folder[0], file), 'meta': os.path.join(folder[0], meta)})
    return dataset_paths

In [8]:
def create_full_dataset_dataframe(year = None):
    dataset_paths = get_dataset_paths(year)
    full_dataset_dataframe = pd.DataFrame()
    for pair in dataset_paths:
        conllu_path = pair['conllu']
        metadata_path = pair['meta']
        metadata_dataframe = get_metadata_dataframe(metadata_path)
        conllu_file = conllu.load_from_file(conllu_path)
        file_with_join_keys = create_join_keys(conllu_file)
        conllu_dataframe = file_to_token_array_dataframe(file_with_join_keys)
        joint_dataframe = create_joint_dataframe(conllu_dataframe, metadata_dataframe)
        full_dataset_dataframe = pd.concat([full_dataset_dataframe, joint_dataframe])
    return full_dataset_dataframe


In [9]:
token_dataframe = create_full_dataset_dataframe(year = None)

Remove rows where the speaker is not an MP (for instance, invited guests or the President who is not a member of parliament) and rows without a party orientation (~1000 rows). Filter very short texts, as these are mostly uninformative or greetings/customary phrases.

In [10]:
def filter_dataframe(df):
    filtered = df[df['Speaker_MP'] == 'MP']
    filtered = filtered[filtered['Speaker_role'] == 'Regular']
    filtered = filtered[filtered['Party_orientation'] != '-']
    filtered = filtered[filtered['length'] > 5]
    return filtered

In [11]:
mp_dataframe = filter_dataframe(token_dataframe)

In [12]:
mp_dataframe.groupby('Party_orientation').count()

,tokens,lemmas,join_key,length,Date,Speaker_role,Speaker_MP,Speaker_minister,Speaker_party,Speaker_ID,Speaker_name
Party_orientation,,,,,,,,,,,
Centre,9113,9113,9113,9113,9113,9113,9113,9113,9113,9113,9113
Centre to centre-left,116326,116326,116326,116326,116326,116326,116326,116326,116326,116326,116326
Centre to centre-right,2140,2140,2140,2140,2140,2140,2140,2140,2140,2140,2140
Centre-left,239590,239590,239590,239590,239590,239590,239590,239590,239590,239590,239590
Centre-left to left,26813,26813,26813,26813,26813,26813,26813,26813,26813,26813,26813
Centre-right,204621,204621,204621,204621,204621,204621,204621,204621,204621,204621,204621
Far-right,11387,11387,11387,11387,11387,11387,11387,11387,11387,11387,11387
Right,119070,119070,119070,119070,119070,119070,119070,119070,119070,119070,119070
Right to far-right,290861,290861,290861,290861,290861,290861,290861,290861,290861,290861,290861


In [13]:
sampled_token_dataframe = mp_dataframe.sample(frac = 0.5)

In [14]:
sampled_token_dataframe.groupby('Party_orientation').count()

,tokens,lemmas,join_key,length,Date,Speaker_role,Speaker_MP,Speaker_minister,Speaker_party,Speaker_ID,Speaker_name
Party_orientation,,,,,,,,,,,
Centre,4635,4635,4635,4635,4635,4635,4635,4635,4635,4635,4635
Centre to centre-left,57889,57889,57889,57889,57889,57889,57889,57889,57889,57889,57889
Centre to centre-right,1083,1083,1083,1083,1083,1083,1083,1083,1083,1083,1083
Centre-left,120018,120018,120018,120018,120018,120018,120018,120018,120018,120018,120018
Centre-left to left,13470,13470,13470,13470,13470,13470,13470,13470,13470,13470,13470
Centre-right,102519,102519,102519,102519,102519,102519,102519,102519,102519,102519,102519
Far-right,5717,5717,5717,5717,5717,5717,5717,5717,5717,5717,5717
Right,59307,59307,59307,59307,59307,59307,59307,59307,59307,59307,59307
Right to far-right,145322,145322,145322,145322,145322,145322,145322,145322,145322,145322,145322


In [15]:
sampled_token_dataframe.to_pickle('./sampled_dataframe.pkl')

In [16]:
mp_dataframe.to_pickle('./unsampled_dataframe.pkl')